In [7]:
import os
import pandas as pd
import plotly.graph_objs as go
from plotly.subplots import make_subplots
from pathlib import Path

# Paths
cached_spectrums = Path("./cached_spectrums")

# Load precomputed medians from CSV
median_df = pd.read_csv(cached_spectrums / "median_snr_all.csv")

# Voltage types and their column names
voltage_types = [
    ("mininghighvoltage", "Mining High Voltage"),
    ("mininglowvoltage", "Mining Low Voltage"),
    ("soilhighvoltage", "Soil High Voltage"),
    ("soilmidvoltage", "Soil Mid Voltage"),
    ("soillowvoltage", "Soil Low Voltage")
]

# X-ray line catalog (keV)
LINE_CATALOG = {
    "elements": {
        # --- K lines (kept where <= 30 keV) ---
        "Mg_Ka": 1.2536, "Mg_Kb": 1.302,
        "Al_Ka": 1.4867, "Al_Kb": 1.550,
        "Si_Ka": 1.7400, "Si_Kb": 1.836,
        "P_Ka": 2.0137, "P_Kb": 2.139,
        "S_Ka": 2.3070, "S_Kb": 2.464,
        "Cl_Ka": 2.6220, "Cl_Kb": 2.815,
        "K_Ka": 3.3120, "K_Kb": 3.590,
        "Ca_Ka": 3.6910, "Ca_Kb": 4.012,
        "Ti_Ka": 4.5108, "Ti_Kb": 4.9318,
        "V_Ka": 4.9520, "V_Kb": 5.427,
        "Cr_Ka": 5.4150, "Cr_Kb": 5.946,
        "Mn_Ka": 5.8988, "Mn_Kb": 6.490,
        "Fe_Ka": 6.4038, "Fe_Kb": 7.058,
        "Co_Ka": 6.9300, "Co_Kb": 7.649,
        "Ni_Ka": 7.4780, "Ni_Kb": 8.265,
        "Cu_Ka": 8.0480, "Cu_Kb": 8.905,
        "Zn_Ka": 8.6389, "Zn_Kb": 9.572,
        "Ga_Ka": 9.2510, "Ga_Kb": 10.264,
        "Ge_Ka": 9.8860, "Ge_Kb": 10.982,
        "As_Ka": 10.5430, "As_Kb": 11.723,
        "Se_Ka": 11.2220, "Se_Kb": 12.486,
        "Br_Ka": 11.9240, "Br_Kb": 13.272,
        "Rb_Ka": 13.3950, "Rb_Kb": 14.829,
        "Sr_Ka": 14.1650, "Sr_Kb": 15.682,
        "Y_Ka": 14.9580, "Y_Kb": 16.738,
        "Zr_Ka": 15.7750, "Zr_Kb": 17.667,
        "Nb_Ka": 16.6150, "Nb_Kb": 18.629,
        "Mo_Ka": 17.4790, "Mo_Kb": 19.608,
        "Ru_Ka": 19.2780, "Ru_Kb": 21.654,
        "Rh_Ka": 20.2160, "Rh_Kb": 22.747,
        "Pd_Ka": 21.1740, "Pd_Kb": 23.864,
        "Ag_Ka": 22.1630, "Ag_Kb": 25.013,
        "Cd_Ka": 23.1730, "Cd_Kb": 26.187,
        "In_Ka": 24.2080, "In_Kb": 27.389,
        "Sn_Ka": 25.2710, "Sn_Kb": 28.619,
        "Sb_Ka": 26.3590, "Sb_Kb": 29.877,
        "Te_Ka": 27.4720,
        "I_Ka": 28.6120,
        "Ba_Ka": 29.7790,

        # --- L lines (heavy elements, better below 30 keV) ---
        "Zr_La": 2.042, "Zr_Lb": 2.307,
        "Nb_La": 2.166, "Nb_Lb": 2.443,
        "Mo_La": 2.293, "Mo_Lb": 2.629,
        "Ru_La": 2.558, "Ru_Lb": 2.683,
        "Rh_La": 2.697, "Rh_Lb": 2.834,
        "Pd_La": 2.839, "Pd_Lb": 2.990,
        "Ag_La": 2.984, "Ag_Lb": 3.150,
        "Cd_La": 3.133, "Cd_Lb": 3.316,
        "In_La": 3.286, "In_Lb": 3.487,
        "Sn_La": 3.444, "Sn_Lb": 3.670,
        "Sb_La": 3.605, "Sb_Lb": 3.843,
        "Te_La": 3.769, "Te_Lb": 4.029,
        "I_La": 3.938, "I_Lb": 4.220,
        "Ba_La": 4.466, "Ba_Lb": 4.828,
        "La_La": 4.650, "La_Lb": 5.043,
        "Ce_La": 4.840, "Ce_Lb": 5.262,
        "Pr_La": 5.033, "Pr_Lb": 5.489,
        "Nd_La": 5.230, "Nd_Lb": 5.722,
        "Sm_La": 5.637, "Sm_Lb": 6.180,
        "Eu_La": 5.846, "Eu_Lb": 6.456,
        "Gd_La": 6.056, "Gd_Lb": 6.713,
        "Tb_La": 6.273, "Tb_Lb": 6.978,
        "Dy_La": 6.495, "Dy_Lb": 7.247,
        "Ho_La": 6.719, "Ho_Lb": 7.526,
        "Er_La": 6.949, "Er_Lb": 7.810,
        "Tm_La": 7.180, "Tm_Lb": 8.102,
        "Yb_La": 7.416, "Yb_Lb": 8.401,
        "Lu_La": 7.655, "Lu_Lb": 8.710,
        "Hf_La": 7.899, "Hf_Lb": 9.022,
        "Ta_La": 8.146, "Ta_Lb": 9.343,
        "W_La": 8.398, "W_Lb": 9.673,
        "Re_La": 8.652, "Re_Lb": 10.010,
        "Os_La": 8.911, "Os_Lb": 10.355,
        "Ir_La": 9.175, "Ir_Lb": 10.708,
        "Pt_La": 9.442, "Pt_Lb": 11.072,
        "Au_La": 9.713, "Au_Lb": 11.442,
        "Hg_La": 9.989, "Hg_Lb": 11.822,
        "Tl_La": 10.269, "Tl_Lb": 12.213,
        "Pb_La": 10.551, "Pb_Lb": 12.614,
        "Bi_La": 10.839, "Bi_Lb": 13.023,
    },
    "instrument": {}
}

# Get all target names from the median file
target_names = median_df['target'].unique()

# Assign a unique color to each target for consistent legend/plot color
import plotly.colors as pc
palette = pc.qualitative.Plotly
color_map = {name: palette[i % len(palette)] for i, name in enumerate(target_names)}

# Plotly: 5 stacked plots, one per voltage
fig = make_subplots(rows=5, cols=1, shared_xaxes=True, vertical_spacing=0.03,
                    subplot_titles=[v[1] for v in voltage_types])

# For legend: only one entry per target (first voltage type)
legend_targets = set()

# Add traces: for each target, add all voltages in the same order, and group bands with their median line using legendgroup. Only show legend for the first voltage for each target.
for j, target_name in enumerate(target_names):
    color = color_map[target_name]
    for i, (vcol, vname) in enumerate(voltage_types, 1):
        vdf = median_df[(median_df['target'] == target_name) & (median_df['voltage'] == vcol)]
        if vdf.empty or not all(col in vdf.columns for col in ['q05','q25','q75','q95']):
            continue
        energy_axis = vdf['Energy (keV)'].values
        # 25-75 band
        x_band1 = list(energy_axis) + list(energy_axis[::-1])
        y_band1 = list(vdf['q25'].values) + list(vdf['q75'].values[::-1])
        # 5-95 band
        x_band2 = list(energy_axis) + list(energy_axis[::-1])
        y_band2 = list(vdf['q05'].values) + list(vdf['q95'].values[::-1])
        showlegend = False
        if target_name not in legend_targets:
            showlegend = True
            legend_targets.add(target_name)
        # Add bands (hidden from legend, but grouped with median line)
        fig.add_trace(
            go.Scatter(
                x=x_band2,
                y=y_band2,
                fill='toself',
                fillcolor='rgba(0,100,200,0.07)',
                line=dict(width=0),
                showlegend=False,
                name=f'{target_name} Q05-Q95',
                hoverinfo='skip',
                legendgroup=target_name,
            ),
            row=i, col=1
)
        fig.add_trace(
            go.Scatter(
                x=x_band1,
                y=y_band1,
                fill='toself',
                fillcolor='rgba(0,100,200,0.15)',
                line=dict(width=0),
                showlegend=False,
                name=f'{target_name} Q25-Q75',
                hoverinfo='skip',
                legendgroup=target_name,
            ),
            row=i, col=1
)
        # Median line (main legend entry)
        fig.add_trace(
            go.Scatter(
                x=energy_axis,
                y=vdf['SNR_median'],
                name=target_name,
                mode='lines',
                legendgroup=target_name,
                showlegend=showlegend,
                line=dict(color=color)
            ),
            row=i, col=1
)

# Add all X-ray lines as a single Scatter trace with hover labels, 1 SNR tall
for i, (vcol, vname) in enumerate(voltage_types, 1):
    x_lines = []
    y_lines = []
    text_lines = []
    for label, energy in list(LINE_CATALOG['elements'].items()) + list(LINE_CATALOG['instrument'].items()):
        x_lines.extend([energy, energy, None])
        y_lines.extend([0, 1, None])
        text_lines.extend([label, label, None])
    fig.add_trace(
        go.Scatter(
            x=x_lines,
            y=y_lines,
            mode='lines',
            line=dict(color='gray', dash='dot', width=1),
            showlegend=False,
            text=text_lines,
            hoverinfo='text'
)
        ,row=i, col=1
)

# Set keV ticks and labels on all plots, but only show x-axis title on the bottom plot
tickvals = list(range(0, int(max(median_df['Energy (keV)']))+5, 5))
ticktext = [str(x) for x in tickvals]
for i, (vcol, vname) in enumerate(voltage_types, 1):
    if i == len(voltage_types):
        # Only bottom plot gets the x-axis title
        fig.update_xaxes(
            title_text='Energy (keV)',
            tickmode='array',
            tickvals=tickvals,
            ticktext=ticktext,
            row=i, col=1,
            showticklabels=True,
            showline=True,
            mirror=True,
            ticks='outside',
            ticklen=8,
            tickwidth=2,
            tickcolor='black'
)
    else:
        fig.update_xaxes(
            title_text='',
            tickmode='array',
            tickvals=tickvals,
            ticktext=ticktext,
            row=i, col=1,
            showticklabels=True,
            showline=True,
            mirror=True,
            ticks='outside',
            ticklen=8,
            tickwidth=2,
            tickcolor='black'
)
    fig.update_yaxes(title_text="Median SNR", row=i, col=1)
fig.update_layout(
    height=1800,
    showlegend=True,
    title="Median SNR for Each Target (per Voltage)",
    legend_title_text="Target",
)


fig.show(renderer="browser")


In [8]:

# Optional HTML export
export_html = True
html_output_path = cached_spectrums / "median_snr_targets.html"
if export_html:
    fig.write_html(html_output_path, include_plotlyjs='cdn')
    print(f"Saved HTML plot to: {html_output_path.resolve()}")

Saved HTML plot to: C:\Users\phuynh\Projects\robotray-main\plots_phan\cached_spectrums\median_snr_targets.html
